# ModernBERT-JEPA decider, large: the answer points at an option

Run 11 (`05_jepa_point.ipynb`, ModernBERT-base) found that the JEPA pointer lifts JevBench's public items from 0.394 (run 8 B) to 0.493, easy from 0.56 to 0.83. This run is the same arm C2 on **ModernBERT-large** (395M), to see how far the encoder size carries it; it is the encoder to compare with ~1B decoders.

The method is unchanged from run 11:
- **Context side:** the encoder reads `[CLS] question [SEP] state [SEP] A: option … Answer: [MASK]`; the predictor turns the `[MASK]` vector into ẑ.
- **Target side:** the EMA copy of the encoder reads the same input; its vectors at the option label tokens (`A`, `yes`, `7`, …) are the targets.
- **JEPA loss:** ẑ is pulled onto the gold option's label vector (1 − cos, weighted by the gold distribution).
- **Pointer:** softmax over the question's own labels of cos(ẑ, v_k) / τ; every other label is 0. KL to the gold.
- **The 38-way head** as in run 8; C2's decision averages the head's and the pointer's probabilities.

**What changes for large on two T4s:**
- **Gradient checkpointing** in the encoder: activations are recomputed in the backward pass, so the model, its EMA copy and the optimizer fit (about 10–12 GB on the first GPU).
- **One pass** over the mixed data by default (`ARM_EPOCHS`): run 11's C2 was already at 0.489 on JevBench after one pass, and two passes of large would take about 10 hours.
- **C2 only** by default (`ARMS`); the control is run 8 B and run 11's C2 on base.
- **The same tasksource sample** as run 11, read from its cache in `/kaggle/working/ckpt05` when it is there.
- **Disk:** a checkpoint of large is about 6.5 GB (both encoders and the optimizer), and saving briefly needs twice that; the free space is checked before training.

**Checkpoints** as in run 11: every `CKPT_EVERY` steps and after each quarter's scoring, to `CKPT_DIR`; after a crash, restart the kernel and run all cells. Its final weights (`C2_final.pt`) can be scored with `06_jevbench_cli.ipynb` (set `ENCODER` and `WEIGHTS` there).

## Parameters

In [ ]:
import os

def env(name, default):
    return os.environ.get(name, default)

ENCODER = env("ENCODER", "answerdotai/ModernBERT-large")
DATASET = "skamalj/signpost-d-decisions"           # our generated data (private): train and dev
JEVBENCH_COMMIT = "609a664e81f46e03731600bd7bd9f1b7a61257c5"   # pinned public items
TASKSOURCE = "tasksource/tasksource-jev-typed-decisions"       # public typed decisions
TS_QUESTIONS = int(env("TS_QUESTIONS", "50000"))   # tasksource sample size
TS_PER_SOURCE = int(env("TS_PER_SOURCE", "400"))   # at most this many from one source
TS_BUFFER = 50000                                  # streaming shuffle buffer
OVERLAP_N = 8                                      # drop a state sharing an 8-word run with a test
TRAIN_QUESTIONS = int(env("TRAIN_QUESTIONS", "0"))  # our questions to train on; 0 = all 12,274
MAX_OPTIONS = 26                                   # letters A-Z
BATCH = int(env("BATCH", "16"))                    # questions per step
TRAIN_MAX_LEN = 1024
EVAL_MAX_LEN = 8192                                # JevBench's hard items are long documents
OPTION_MAX_TOKENS = 64                             # each option's text is cut to this
EVAL_TOKEN_BUDGET = 16384                          # tokens per evaluation batch
LR_ENCODER = float(env("LR_ENCODER", "5e-5"))      # top layer; lower layers decay by LLRD
LLRD = float(env("LLRD", "0.95"))                  # gentle, so the whole encoder trains
LR_HEAD = float(env("LR_HEAD", "5e-4"))            # label head, predictor, temperature
ARM_EPOCHS = int(env("ARM_EPOCHS", "1"))           # passes over the training data
CURVE_POINTS = 4                                   # scored at each quarter of an arm's training
WARMUP = 0.05
EMA_START, EMA_END = 0.996, 0.9999                 # target-encoder momentum, cosine schedule
LAMBDA_JEPA, LAMBDA_POINT, LAMBDA_ORD = 0.5, 1.0, 0.5
TAU_START = 0.1                                    # pointer temperature (learned)
ARMS = env("ARMS", "C2").split(",")                # C2 only; C1 = head only, if wanted
CKPT_DIR = env("CKPT_DIR", "/kaggle/working/ckpt07" if os.path.isdir("/kaggle/working")
               else "ckpt07")
TS_CACHE_DIRS = [CKPT_DIR, "/kaggle/working/ckpt05"]  # run 11's tasksource sample, if there
GRAD_CHECKPOINTING = True                          # recompute activations: large fits on a T4
MIN_FREE_GB = 15                                   # a checkpoint and its temporary copy
CKPT_EVERY = int(env("CKPT_EVERY", "500"))         # steps between checkpoints
SEED = int(env("SEED", "42"))
os.makedirs(CKPT_DIR, exist_ok=True)

## Packages and the Hugging Face token

The token reads our private dataset: from the environment, a Kaggle or Colab secret, or a hidden prompt. Only where it came from is printed.

In [ ]:
!pip install -q "transformers>=4.48"

import copy, gc, json, math, random, re, shutil, statistics, time, urllib.request
from collections import Counter, defaultdict

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    import psutil                       # host memory in the training log
except ImportError:
    psutil = None


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle secret"
    except Exception:
        pass
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token, "Colab secret"
    except Exception:
        pass
    import getpass
    token = getpass.getpass("Hugging Face token (read access): ").strip()
    return (token, "the prompt") if token else (None, "NOT FOUND")


HF_TOKEN, token_from = hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face token: {token_from}")

## Device and log

One or more GPUs: with two, every batch is split across both. Mixed precision is bf16 where the GPU has it and fp16 with loss scaling otherwise (a T4).

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
AMP = DEVICE.type == "cuda"
# bf16 only with bf16 hardware (compute capability 8.0+: A100, G4); a T4 reports bf16 as
# supported but only emulates it, very slowly, so it uses fp16 with loss scaling
BF16_HW = AMP and torch.cuda.get_device_capability(0)[0] >= 8
AMP_DTYPE = torch.bfloat16 if BF16_HW else torch.float16


def amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP)


def new_scaler():
    """Loss scaling, needed for fp16 only (bf16 has fp32's range)."""
    return torch.amp.GradScaler("cuda", enabled=AMP and AMP_DTYPE == torch.float16)


def spread(module):
    """With several GPUs, each batch is split across them (data parallel)."""
    return nn.DataParallel(module) if N_GPU > 1 else module


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


gpu = f"{N_GPU} x {torch.cuda.get_device_name()}" if AMP else "CPU"
log(f"torch {torch.__version__} | transformers {transformers.__version__} | {gpu} | "
    f"{str(AMP_DTYPE).split('.')[-1] if AMP else 'fp32'} | {ENCODER}")

## One item per question

Every question becomes: question text, state text, option texts and a gold distribution over the options. `noul` options are the criteria's `true` / `false` descriptions (or nothing), shown as `yes` / `no`; `choice` options are `name - description`, shown as `A`, `B`, …; `score` levels are shown as `0`, `1`, …

In [ ]:
def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def option_texts(q):
    """-> (option names, option texts) for one System One question."""
    kind, crit = q["type"], q.get("criteria")
    if kind == "noul":
        crit = crit or {}
        return ["true", "false"], [crit.get("true") or "", crit.get("false") or ""]
    if kind == "choice":
        names = list(crit)
        return names, [f"{n} - {d}" if d else n for n, d in crit.items()]
    return [str(i) for i in range(len(crit))], [str(level) for level in crit]


def make_item(kind, question, state, q, gold, source, family=""):
    names, texts = option_texts(q)
    total = sum(gold)
    return {"type": kind, "question": question or "", "state": state_text(state), "names": names,
            "texts": texts, "gold": [g / total for g in gold], "source": source, "family": family}

## Our training and dev data

In [ ]:
def load_split(split):
    path = hf_hub_download(DATASET, f"data/{split}.jsonl", repo_type="dataset", token=HF_TOKEN)
    items = []
    for line in open(path, encoding="utf-8"):
        rec = json.loads(line)
        for qid, q in rec["questions"].items():
            probs = rec["gold"][qid]["probabilities"]
            names, _ = option_texts(q)
            if len(names) > MAX_OPTIONS:
                continue
            gold = [float(probs[n]) for n in names]
            items.append(make_item(q["type"], q.get("instructions"), rec["state"], q, gold, split))
    return items


train_all, dev_items = load_split("train"), load_split("dev")
train_items = (random.Random(SEED).sample(train_all, min(TRAIN_QUESTIONS, len(train_all)))
               if TRAIN_QUESTIONS else train_all)
print(f"train {len(train_items)} of {len(train_all)} questions "
      f"{dict(Counter(it['type'] for it in train_items))} | dev {len(dev_items)}")

## JevBench public items (test only, never trained on)

One question per item and one expected label, so the gold is one-hot.

In [ ]:
def load_jevbench(split):
    url = (f"https://raw.githubusercontent.com/fstandhartinger/jevbench/{JEVBENCH_COMMIT}"
           f"/datasets/public/{split}.jsonl")
    items = []
    for line in urllib.request.urlopen(url).read().decode("utf-8").splitlines():
        rec = json.loads(line)
        q = rec["question"]
        names, _ = option_texts(q)
        if q["type"] == "noul":
            target = "true" if rec["expected"] == "yes" else "false"
        else:
            target = str(rec["expected"])
        gold = [1.0 if n == target else 0.0 for n in names]
        assert sum(gold) == 1, rec["id"]
        items.append(make_item(q["type"], q["instructions"], rec["state"], q, gold,
                               split, rec["family"]))
    return items


jev_items = [it for split in ("original", "easy", "hard") for it in load_jevbench(split)]
print(f"JevBench public: {len(jev_items)} items {dict(Counter(it['source'] for it in jev_items))} "
      f"{dict(Counter(it['type'] for it in jev_items))}")

## Stage 1 data: public typed decisions, filtered

A guard set of word 8-grams from every test state we care about (JevBench's public items, Typed Decisions' train and test splits). Then the tasksource stream, shuffled: rows whose `license_use` is commercial, with 2–26 options (2–10 levels for `score`), at most `TS_PER_SOURCE` per source, and no 8-gram shared with the guard set.

In [ ]:
WORD = re.compile(r"\w+")


def ngrams(text, n=OVERLAP_N):
    words = WORD.findall(text.lower())
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


guard = set()
for it in jev_items:
    guard |= ngrams(it["state"])
for split in ("train", "test"):
    path = hf_hub_download("LocalLLaMA/typed-decisions", f"all/{split}-00000-of-00001.parquet",
                           repo_type="dataset")
    for state in pd.read_parquet(path)["state"]:
        guard |= ngrams(state_text(json.loads(state)))
print(f"guard set: {len(guard):,} 8-grams")

## Tasksource sample (cached)

The tasksource stream, shuffled: rows whose `license_use` is commercial, with 2–26 options (2–10 levels for `score`), at most `TS_PER_SOURCE` per source, and no 8-gram shared with the guard set. The sample is saved in `CKPT_DIR`; a rerun reads it from there, or from run 11's `ckpt05` (the same sample, so the data match run 11).

In [ ]:
def ts_item(row):
    """A tasksource row -> one item, or None if it does not fit."""
    kind, opts, target = row["kind"], row["options"], row["target"]
    if kind == "noul":
        q, gold = {"type": "noul", "criteria": None}, [target[0], 1 - target[0]]
    elif kind == "choice" and 2 <= len(opts) <= MAX_OPTIONS and len(set(opts)) == len(opts):
        q, gold = {"type": "choice", "criteria": {o: None for o in opts}}, list(target)
    elif kind == "score" and 2 <= len(opts) <= 10:
        q, gold = {"type": "score", "criteria": list(opts)}, list(target)
    else:
        return None
    if sum(gold) <= 0 or len(gold) != len(option_texts(q)[0]):
        return None
    return make_item(kind, row["question"], row["state"], q, gold, "tasksource", row["source"])


def sample_tasksource():
    items, per_source, dropped = [], Counter(), Counter()
    stream = load_dataset(TASKSOURCE, "default", split="train", streaming=True, token=HF_TOKEN)
    for row in stream.shuffle(seed=SEED, buffer_size=TS_BUFFER):
        if row["license_use"] != "commercial":
            dropped["licence"] += 1
            continue
        if per_source[row["source"]] >= TS_PER_SOURCE:
            continue
        item = ts_item(row)
        if item is None:
            dropped["shape"] += 1
            continue
        if ngrams(item["state"]) & guard:
            dropped["overlap"] += 1
            continue
        items.append(item)
        per_source[row["source"]] += 1
        if len(items) >= TS_QUESTIONS:
            break
    print(f"sampled from {len(per_source)} sources | dropped {dict(dropped)}")
    return items


ts_name = f"tasksource_{TS_QUESTIONS}_{TS_PER_SOURCE}_{SEED}.json"
ts_cache = next((os.path.join(d, ts_name) for d in TS_CACHE_DIRS
                 if os.path.exists(os.path.join(d, ts_name))), os.path.join(CKPT_DIR, ts_name))
if os.path.exists(ts_cache):
    ts_items = json.load(open(ts_cache, encoding="utf-8"))
    print(f"read from {ts_cache}")
else:
    ts_items = sample_tasksource()
    json.dump(ts_items, open(ts_cache, "w", encoding="utf-8"))
print(f"tasksource: {len(ts_items)} questions {dict(Counter(it['type'] for it in ts_items))}")

## Label tokens

The 38 labels are A–Z, yes / no and 0–9. With the leading space they have after `Answer:`, each must be one token of ModernBERT's vocabulary: the head starts from those tokens' pretrained output rows. At the start of an option line (`A: …`) a label must keep its own tokens, so its position can be found; both are checked here.

In [ ]:
tok = AutoTokenizer.from_pretrained(ENCODER)
CLS, SEP, PAD, MASK = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id, tok.mask_token_id
LABELS = {"choice": [chr(65 + i) for i in range(MAX_OPTIONS)],
          "noul": ["yes", "no"],
          "score": [str(i) for i in range(10)]}


def ids(text):
    return tok(text, add_special_tokens=False)["input_ids"]


def one_token(label):
    found = ids(" " + label)
    assert len(found) == 1, f"label {label!r} is {len(found)} tokens"
    return found[0]


LABEL_IDS = {kind: [one_token(lab) for lab in labs] for kind, labs in LABELS.items()}
HEAD_TOKENS = LABEL_IDS["choice"] + LABEL_IDS["noul"] + LABEL_IDS["score"]   # the 38 head outputs
HEAD_SLOTS = {"choice": list(range(26)), "noul": [26, 27], "score": list(range(28, 38))}
for labs in LABELS.values():
    for lab in labs:
        assert ids(f"{lab}: x")[:len(ids(lab))] == ids(lab), f"label {lab!r} merges with ':'"
print({k: v[:4] for k, v in LABEL_IDS.items()}, "| head outputs", len(HEAD_TOKENS))

## One sequence per question

`[CLS] question [SEP] state [SEP] A: option … [SEP] Answer: [MASK] [SEP]`. The question and the options are kept whole (each option cut at `OPTION_MAX_TOKENS`); if the sequence is too long, the state's end is cut, and the cut is counted. Also returned: the position of each option's label token (the pointer's targets). In training, choice options are shuffled, so a letter never carries a fixed answer.

In [ ]:
def shuffled(item, rng):
    """A copy of a choice item with its options in a random order (gold moved with them)."""
    if item["type"] != "choice" or rng is None:
        return item
    order = list(range(len(item["names"])))
    rng.shuffle(order)
    return {**item, "names": [item["names"][i] for i in order],
            "texts": [item["texts"][i] for i in order], "gold": [item["gold"][i] for i in order]}


def sequence(item, max_len):
    """-> (token ids with [MASK], mask position, label positions, state was cut)."""
    labels = LABELS[item["type"]]
    tail, at = [], []
    for label, text in zip(labels, item["texts"]):
        line = f"{label}: {text}".strip() if text else label
        at.append(len(tail) + len(ids(label)) - 1)        # the label's (last) token
        tail += ids(line)[:OPTION_MAX_TOKENS] + [SEP]
    tail += ids("Answer:")
    q = ids(item["question"])[:256]
    s = ids(item["state"])
    room = max_len - len(q) - len(tail) - 5
    cut = len(s) > room
    s = s[:max(room, 0)]
    head = [CLS] + q + [SEP] + s + [SEP]
    seq = head + tail
    pos = len(seq)
    seq += [MASK, SEP]
    return seq, pos, [len(head) + a for a in at], cut


def batch_tensors(items, max_len):
    rows = [sequence(it, max_len) for it in items]
    width = max(len(r[0]) for r in rows)
    k = max(len(r[2]) for r in rows)
    input_ids, att, pos, label_pos = [], [], [], []
    for seq, p, at, _ in rows:
        input_ids.append(seq + [PAD] * (width - len(seq)))
        att.append([1] * len(seq) + [0] * (width - len(seq)))
        pos.append(p)
        label_pos.append(at + [0] * (k - len(at)))        # padding points at [CLS]; masked out
    as_t = lambda x: torch.tensor(x, device=DEVICE)
    return as_t(input_ids), as_t(att), as_t(pos), as_t(label_pos), sum(r[3] for r in rows)

## The model

- **Encoder** (trained): ModernBERT-large, with gradient checkpointing; its pretrained MLM transform turns the `[MASK]` vector into the space of token outputs.
- **Label head** (trained): 38 outputs (A–Z, yes / no, 0–9), started from the pretrained output rows of those tokens; each question keeps only its own labels.
- **Predictor** (trained): an MLP with a residual from the `[MASK]` vector to ẑ.
- **Target encoder** (EMA copy of the encoder, no gradients, always in eval mode): reads the same input; its vectors at the option label tokens are the targets. It is also used to score the pointer.
- **Temperature τ** (trained) for the pointer.

In [ ]:
TYPES = ["noul", "choice", "score"]


class EncoderAt(nn.Module):
    """An encoder's vectors at given positions: one per row ([rows]) or several ([rows, k]);
    split across GPUs by spread()."""

    def __init__(self, encoder):
        super().__init__()
        self.enc = encoder

    def forward(self, input_ids, att, pos):
        with amp():                                   # each GPU's copy runs in mixed precision
            h = self.enc(input_ids=input_ids, attention_mask=att).last_hidden_state
        rows = torch.arange(len(pos), device=h.device)
        return h[rows, pos] if pos.dim() == 1 else h[rows[:, None], pos]


class MaskedDecider(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(ENCODER, attn_implementation="sdpa")
        if GRAD_CHECKPOINTING:                        # only active in training mode
            self.mlm.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False})
        hidden = self.mlm.config.hidden_size
        for p in self.mlm.model.embeddings.parameters():
            p.requires_grad = False
        self.predictor = nn.Sequential(nn.Linear(hidden, 2 * hidden), nn.GELU(),
                                       nn.LayerNorm(2 * hidden), nn.Linear(2 * hidden, hidden))
        self.label_head = nn.Linear(hidden, len(HEAD_TOKENS))
        with torch.no_grad():                         # start from the pretrained output rows
            self.label_head.weight.copy_(self.mlm.decoder.weight[HEAD_TOKENS])
            if self.mlm.decoder.bias is not None:
                self.label_head.bias.copy_(self.mlm.decoder.bias[HEAD_TOKENS])
        self.log_tau = nn.Parameter(torch.tensor(math.log(TAU_START)))
        self.target = copy.deepcopy(self.mlm.model).requires_grad_(False)
        self.at_mask = spread(EncoderAt(self.mlm.model))
        self.target_at = spread(EncoderAt(self.target))

    def mask_vectors(self, input_ids, att, pos):
        return self.at_mask(input_ids, att, pos)

    def label_logits(self, vectors, items):
        """The 38-way head at [MASK]; each question keeps its own labels -> [items, options]."""
        all38 = self.label_head(self.mlm.head(vectors)).float()
        width = max(len(it["names"]) for it in items)
        out = torch.full((len(items), width), -1e4, device=vectors.device)
        for i, it in enumerate(items):
            k = len(it["names"])
            out[i, :k] = all38[i, HEAD_SLOTS[it["type"]][:k]]
        return out

    def tau(self):
        return self.log_tau.exp().clamp(min=0.01)

    def train(self, mode=True):
        """Train the online parts; the EMA target encoder always stays in eval mode."""
        super().train(mode)
        self.target.eval()
        return self

    @torch.no_grad()
    def target_vectors(self, input_ids, att, label_pos):
        return self.target_at(input_ids, att, label_pos)

## Scores and losses for a batch

Scores per readout: the head's, and with the pointer on, cos(ẑ, v_k) / τ over the question's own labels. Losses: KL to the gold for the head and for the pointer; the ordinal error for `score` (head); JEPA, 1 − cos(ẑ, v_k) weighted by the gold distribution. The probabilities: the head's, the pointer's, and both averaged (C2's decision).

In [ ]:
def forward(model, items, max_len, pointer):
    input_ids, att, pos, label_pos, cut = batch_tensors(items, max_len)
    v = model.mask_vectors(input_ids, att, pos)
    scores, extra = {"head": model.label_logits(v, items)}, {}
    if pointer:
        targets = model.target_vectors(input_ids, att, label_pos).float()   # [items, k, hidden]
        z = (v + model.predictor(v)).float()
        cos = F.cosine_similarity(z.unsqueeze(1), targets, dim=-1)
        k = torch.tensor([len(it["names"]) for it in items], device=cos.device)
        valid = torch.arange(cos.shape[1], device=cos.device)[None] < k[:, None]
        scores["point"] = (cos / model.tau()).masked_fill(~valid, -1e4)
        extra = {"z": z, "targets": targets, "cos": cos, "valid": valid}
    return scores, extra, cut


def gold_matrix(items, width):
    g = torch.zeros(len(items), width, device=DEVICE)
    for i, it in enumerate(items):
        g[i, :len(it["gold"])] = torch.tensor(it["gold"], device=DEVICE)
    return g


def kl(g, scores):
    logp = F.log_softmax(scores, dim=1)
    return (g * (g.clamp(min=1e-12).log() - logp)).sum(1).mean(), logp


def losses(scores, extra, items):
    g = gold_matrix(items, scores["head"].shape[1])
    dec, logp = kl(g, scores["head"])
    parts = {"dec": dec}
    is_score = torch.tensor([it["type"] == "score" for it in items], device=DEVICE)
    if is_score.any():
        levels = torch.arange(g.shape[1], device=DEVICE, dtype=torch.float32)
        n = torch.tensor([len(it["gold"]) - 1 for it in items], device=DEVICE).clamp(min=1)
        gap = ((logp.exp() - g) * levels).sum(1).abs() / n
        parts["ord"] = gap[is_score].mean()
    if "point" in scores:
        parts["point"] = kl(g, scores["point"])[0]
        parts["jepa"] = (g * (1 - extra["cos"])).sum(1).mean()
    weights = {"dec": 1.0, "ord": LAMBDA_ORD, "point": LAMBDA_POINT, "jepa": LAMBDA_JEPA}
    total = sum(weights[k] * v for k, v in parts.items())
    return total, {k: v.item() for k, v in parts.items()}


def readouts(scores):
    """Probabilities per readout: the head, the pointer, and both averaged."""
    probs = {name: F.softmax(s, dim=1) for name, s in scores.items()}
    if "point" in probs:
        probs["both"] = (probs["head"] + probs["point"]) / 2
    return probs

## The measures

JevBench's: accuracy (argmax), multi-class Brier, top-label ECE over 10 equal-width bins (empty bins left out), ordinal MAE for `score`; and KL for our soft dev gold.

In [ ]:
def measure(rows):
    n = len(rows)
    bins = defaultdict(list)
    for r in rows:
        bins[min(int(r["conf"] * 10), 9)].append(r)
    ece = 0.0
    for b in bins.values():
        right = sum(x["ok"] for x in b) / len(b)
        sure = sum(x["conf"] for x in b) / len(b)
        ece += len(b) / n * abs(right - sure)
    out = {"n": n, "acc": round(sum(r["ok"] for r in rows) / n, 4),
           "brier": round(statistics.mean(r["brier"] for r in rows), 4),
           "ece": round(ece, 4), "kl": round(statistics.mean(r["kl"] for r in rows), 4)}
    mae = [r["mae"] for r in rows if r["mae"] is not None]
    if mae:
        out["ordinal_mae"] = round(statistics.mean(mae), 3)
    return out


def row_of(it, p):
    g = it["gold"]
    top, want = max(range(len(p)), key=p.__getitem__), max(range(len(g)), key=g.__getitem__)
    mae = abs(sum(i * x for i, x in enumerate(p)) - want) if it["type"] == "score" else None
    return {"ok": top == want, "conf": p[top], "margin": None,
            "brier": sum((a - b) ** 2 for a, b in zip(p, g)),
            "kl": sum(b * math.log(b / max(a, 1e-9)) for a, b in zip(p, g) if b > 0),
            "mae": mae, "type": it["type"], "source": it["source"], "family": it["family"]}

## Evaluation

Batches by token budget (long items alone), options in their given order. The decision is the head's for C1 and both averaged for C2; the measures overall and by split, type and family, each readout's measures, and the time per item.

In [ ]:
@torch.no_grad()
def evaluate(model, items, max_len, name, pointer):
    model.eval()
    lengths = [len(sequence(it, max_len)[0]) for it in items]
    order = sorted(range(len(items)), key=lengths.__getitem__)
    rows, cut, start, t0 = {}, 0, 0, time.perf_counter()
    while start < len(order):
        batch = [order[start]]
        while (start + len(batch) < len(order)
               and lengths[order[start + len(batch)]] * (len(batch) + 1) <= EVAL_TOKEN_BUDGET):
            batch.append(order[start + len(batch)])
        group = [items[i] for i in batch]
        with amp():
            scores, _, c = forward(model, group, max_len, pointer)
        cut += c
        for readout, probs in readouts(scores).items():
            out = rows.setdefault(readout, [None] * len(items))
            for i, it, p in zip(batch, group, probs.tolist()):
                out[i] = row_of(it, p[:len(it["gold"])])
        start += len(batch)
    ms = (time.perf_counter() - t0) / len(items) * 1000
    main = rows["both" if pointer else "head"]
    result = {"all": measure(main), "readouts": {k: measure(v) for k, v in rows.items()},
              "ms_per_item": round(ms, 1), "states_cut": cut}
    for key in ("source", "type", "family"):
        values = sorted({r[key] for r in main})
        result[key] = {v: measure([r for r in main if r[key] == v]) for v in values}
    log(f"{name}: {result['all']} | {result['ms_per_item']} ms/item | states cut {cut}")
    if pointer:
        log("   readouts: " + " | ".join(f"{k} {v['acc']}" for k, v in result["readouts"].items()))
    for key in ("source", "type"):
        log("   " + " | ".join(f"{k} {v['acc']}" for k, v in result[key].items()))
    return result

## Optimizer, schedule and EMA

Layer-wise learning-rate decay from the top encoder layer; the MLM transform at the encoder rate; the label head, predictor and temperature at a higher rate; warm-up then cosine decay. After every step with JEPA on, the target encoder moves towards the encoder.

In [ ]:
def make_optimizer(model, total_steps):
    layers = model.mlm.model.layers
    groups = [{"params": [p for p in layer.parameters() if p.requires_grad],
               "lr": LR_ENCODER * LLRD ** (len(layers) - 1 - i)} for i, layer in enumerate(layers)]
    in_layers = {id(p) for layer in layers for p in layer.parameters()}
    rest = [p for p in model.mlm.parameters() if p.requires_grad and id(p) not in in_layers]
    groups.append({"params": rest, "lr": LR_ENCODER})
    extra = (list(model.predictor.parameters()) + list(model.label_head.parameters())
             + [model.log_tau])
    groups.append({"params": extra, "lr": LR_HEAD})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    warm = max(1, int(WARMUP * total_steps))

    def factor(s):
        if s < warm:
            return (s + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, factor)


@torch.no_grad()
def ema_update(model, step, total_steps):
    m = EMA_END - (EMA_END - EMA_START) * (math.cos(math.pi * step / total_steps) + 1) / 2
    for pt, ps in zip(model.target.parameters(), model.mlm.model.parameters()):
        pt.mul_(m).add_(ps.detach(), alpha=1 - m)
    return m

## Training

A plan is a list of batches (`ARM_EPOCHS` passes, reshuffled each pass); training runs a slice of it, with choice options shuffled per item, and calls `on_step` every `CKPT_EVERY` steps (the checkpoint). Every 50 steps: the decision loss averaged over the last 50 batches, the current batch's losses, the top learning rate, peak GPU memory and host RAM; with the pointer on, cos(ẑ, gold label vector) against the best other option's, how alike a question's label vectors are, τ, ẑ's norm and the EMA momentum.

In [ ]:
def make_plan(n, epochs, seed):
    """Batches of item indices: `epochs` passes over n items, reshuffled each pass."""
    rng, plan = random.Random(seed), []
    for _ in range(epochs):
        order = list(range(n))
        rng.shuffle(order)
        plan += [order[b:b + BATCH] for b in range(0, n, BATCH)]
    return plan


def memory():
    gpu = max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 2 ** 30 if AMP else 0
    ram = f" RAM {psutil.virtual_memory().percent:.0f}%" if psutil else ""
    return f"GPU peak {gpu:.1f} GB{ram}"


@torch.no_grad()
def pointer_stats(model, extra, items):
    cos, valid = extra["cos"], extra["valid"]
    rows = torch.arange(len(items), device=cos.device)
    gold = torch.tensor([max(range(len(it["gold"])), key=it["gold"].__getitem__) for it in items],
                        device=cos.device)
    other = cos.masked_fill(~valid, -2).scatter(1, gold[:, None], -2).max(1).values
    t = F.normalize(extra["targets"], dim=-1)
    k = cos.shape[1]
    pairs = valid[:, :, None] & valid[:, None, :] & ~torch.eye(k, dtype=torch.bool,
                                                               device=cos.device)
    alike = (t @ t.transpose(1, 2))[pairs].mean()
    return (f" | cos gold {cos[rows, gold].mean().item():.3f} best other {other.mean().item():.3f}"
            f" | label vectors alike {alike.item():.3f} | tau {model.tau().item():.3f}"
            f" | z norm {extra['z'].norm(dim=-1).mean().item():.1f}")


def train_stage(model, opt, sched, state, items, name, pointer, plan, on_step=None):
    model.train()
    recent = []
    rng = random.Random(SEED + len(items) + state["step"])
    t0 = time.perf_counter()
    for chunk in plan:
        batch = [shuffled(items[i], rng) for i in chunk]
        with amp():
            scores, extra, _ = forward(model, batch, TRAIN_MAX_LEN, pointer)
        loss, parts = losses(scores, extra, batch)
        scaler = state["scaler"]
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        params = [p for g in opt.param_groups for p in g["params"]]
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        state["step"] += 1
        m = ema_update(model, state["step"], state["total"]) if pointer else 0.0
        recent = (recent + [parts["dec"]])[-50:]
        if state["step"] % 50 == 0:
            terms = " ".join(f"{k} {v:.4f}" for k, v in parts.items())
            more = pointer_stats(model, extra, batch) + f" | ema {m:.5f}" if pointer else ""
            top_lr = max(g["lr"] for g in opt.param_groups[:-1])
            log(f"{name} step {state['step']}/{state['total']} | decision loss, last 50 "
                f"{statistics.mean(recent):.4f} | batch: {terms} | grad {norm:.2f} | "
                f"lr {top_lr:.1e} | {memory()}{more}")
        if on_step and state["step"] % CKPT_EVERY == 0:
            on_step()
    log(f"{name}: {len(plan)} steps in {(time.perf_counter() - t0) / 60:.1f} min")

## Checkpoints

One file per arm in `CKPT_DIR`: the whole model (encoder, head, predictor, τ and the EMA target), the optimizer, the schedule, the loss scaler, the step and the curve so far. It is written to a temporary file first and then renamed, so a crash while saving leaves the previous checkpoint intact. When an arm finishes, its results go to `<arm>_done.json`, its final weights to `<arm>_final.pt`, and its checkpoint is deleted.

In [ ]:
def ckpt_path(name):
    return os.path.join(CKPT_DIR, f"{name}.pt")


def save_ckpt(name, model, opt, sched, state, curve):
    tmp = ckpt_path(name) + ".tmp"
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": state["scaler"].state_dict(), "step": state["step"], "curve": curve}, tmp)
    os.replace(tmp, ckpt_path(name))
    log(f"{name}: checkpoint at step {state['step']}")


def load_ckpt(name, model, opt, sched, state):
    """Restore an arm from its checkpoint -> its curve so far, or None if there is none."""
    if not os.path.exists(ckpt_path(name)):
        return None
    ck = torch.load(ckpt_path(name), map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"])
    state["scaler"].load_state_dict(ck["scaler"])
    state["step"] = ck["step"]
    return ck["curve"]

## Checks before training

On a small batch from every question type: the target encoder gets no gradients and stays in eval mode after `train()`; its outputs are deterministic; every loss is finite; the answer slot holds only `[MASK]`; each label position holds that label's token; the EMA step moves the target towards the encoder; a checkpoint saved and loaded restores the weights. Then the free disk space for checkpoints, and the parameter counts: the inference model (encoder, MLM transform, label head) apart from the training-only predictor and EMA copy.

In [ ]:
def checks():
    model = MaskedDecider().to(DEVICE)
    sample = [next(it for it in train_items if it["type"] == k) for k in TYPES]
    sample += [next(it for it in ts_items if it["type"] == k) for k in TYPES]
    model.train()
    assert not model.target.training, "target encoder left in training mode"
    assert not any(p.requires_grad for p in model.target.parameters())
    with amp():
        scores, extra, _ = forward(model, sample, TRAIN_MAX_LEN, pointer=True)
        again = forward(model, sample, TRAIN_MAX_LEN, pointer=True)[1]["targets"]
    assert torch.allclose(extra["targets"], again), "target outputs are not deterministic"
    loss, parts = losses(scores, extra, sample)
    assert all(math.isfinite(v) for v in parts.values()), parts
    loss.backward()
    assert all(p.grad is None for p in model.target.parameters()), "target received gradients"
    for it in sample:
        seq, pos, at, _ = sequence(it, TRAIN_MAX_LEN)
        assert seq[pos] == MASK and seq[pos + 1:] == [SEP], "the answer slot is not empty"
        for label, a in zip(LABELS[it["type"]], at):
            assert seq[a] == ids(label)[-1], f"wrong position for label {label!r}"
    pairs = list(zip(model.target.parameters(), model.mlm.model.parameters()))
    p_t, p_o = next((t, o) for t, o in pairs if o.requires_grad)
    with torch.no_grad():
        p_o.add_(0.01)
    before = (p_t - p_o).abs().sum().item()
    ema_update(model, 1, 10)
    after = (p_t - p_o).abs().sum().item()
    assert after < before, "EMA did not move the target towards the encoder"
    opt, sched = make_optimizer(model, 10)
    state = {"step": 3, "total": 10, "scaler": new_scaler()}
    save_ckpt("check", model, opt, sched, state, ["curve"])
    kept = p_o.detach().clone()
    with torch.no_grad():
        p_o.add_(1.0)
    state["step"] = 0
    curve = load_ckpt("check", model, opt, sched, state)
    assert torch.equal(p_o, kept) and state["step"] == 3 and curve == ["curve"], "checkpoint"
    size = os.path.getsize(ckpt_path("check")) / 2 ** 30
    os.remove(ckpt_path("check"))
    free = shutil.disk_usage(CKPT_DIR).free / 2 ** 30
    log(f"checkpoint without optimizer state {size:.1f} GB | free in {CKPT_DIR} {free:.1f} GB")
    assert free >= MIN_FREE_GB, (f"only {free:.1f} GB free; a checkpoint and its temporary copy "
                                 f"need about {MIN_FREE_GB} GB (delete old *_final.pt files?)")
    count = lambda ps: sum(p.numel() for p in ps) / 1e6
    inference = count(model.mlm.parameters()) + count(model.label_head.parameters())
    log(f"checks passed | parts {parts} | inference model {inference:.1f} M | "
        f"predictor {count(model.predictor.parameters()):.1f} M | EMA copy (training only) "
        f"{count(model.target.parameters()):.1f} M")
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()


checks()

## One arm, with a learning curve and checkpoints

From the pretrained weights, or from the arm's checkpoint: one learning-rate schedule over `ARM_EPOCHS` passes; at each quarter, 1,000 of its own training questions, our dev and JevBench. A quarter already scored before a crash is not trained or scored again.

In [ ]:
def point(model, name, label, seen, pointer):
    """(label, training questions, dev, JevBench) at one point of the curve."""
    return (label,
            evaluate(model, seen, TRAIN_MAX_LEN, f"{name} train, {label}", pointer),
            evaluate(model, dev_items, TRAIN_MAX_LEN, f"{name} dev, {label}", pointer),
            evaluate(model, jev_items, EVAL_MAX_LEN, f"{name} JevBench, {label}", pointer))


def run(name, items, pointer):
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider().to(DEVICE)
    seen = random.Random(SEED).sample(items, 1000)
    plan = make_plan(len(items), ARM_EPOCHS, SEED + len(items))
    opt, sched = make_optimizer(model, len(plan))
    state = {"step": 0, "total": len(plan), "scaler": new_scaler()}
    curve = load_ckpt(name, model, opt, sched, state)
    resumed = curve is not None
    if resumed:
        log(f"{name}: resumed at step {state['step']}/{len(plan)}, "
            f"{len(curve) - 1} of {CURVE_POINTS} quarters scored")
    else:
        curve = [point(model, name, "untrained", seen, pointer)]

    def save():
        save_ckpt(name, model, opt, sched, state, curve)

    if not resumed:
        save()
    cuts = [round(len(plan) * k / CURVE_POINTS) for k in range(CURVE_POINTS + 1)]
    for k in range(CURVE_POINTS):
        if len(curve) > k + 1:
            continue
        train_stage(model, opt, sched, state, items, name, pointer,
                    plan[max(cuts[k], state["step"]):cuts[k + 1]], on_step=save)
        curve.append(point(model, name, f"{(k + 1) * ARM_EPOCHS / CURVE_POINTS:g} passes", seen,
                           pointer))
        save()
    return model, curve

## Typed Decisions' train split (never trained on)

Each arm is scored on it at the end. For reference: our 4B decoder 0.720, plain ModernBERT (run 8 B) 0.350, chance 0.318.

In [ ]:
td_path = hf_hub_download("LocalLLaMA/typed-decisions", "all/train-00000-of-00001.parquet",
                          repo_type="dataset")
td_items = []
for row in pd.read_parquet(td_path).to_dict("records"):
    state, qs, gold = (json.loads(row[k]) for k in ("state", "questions", "gold"))
    for qid, q in qs.items():
        names, _ = option_texts(q)
        probs = gold[qid]["probabilities"]
        if len(names) <= MAX_OPTIONS and all(n in probs for n in names):
            td_items.append(make_item(q["type"], q.get("instructions"), state, q,
                                      [float(probs[n]) for n in names], row["workflow"],
                                      f"{row['workflow']}/{qid}"))
print(f"Typed Decisions train: {len(td_items)} questions")

## The arms

The arms in `ARMS` (C2: head + JEPA + pointer; C1: head only), on tasksource and our data mixed. A finished arm's results are read from `CKPT_DIR`; an unfinished arm resumes from its checkpoint.

In [ ]:
mixed = ts_items + train_items
POINTER = {"C1": False, "C2": True}
curves, td_scores = {}, {}
for name in ARMS:
    done = os.path.join(CKPT_DIR, f"{name}_done.json")
    if os.path.exists(done):
        saved = json.load(open(done))
        curves[name], td_scores[name] = saved["curve"], saved["td"]
        log(f"{name}: finished earlier, results read from {done}")
        continue
    model, curves[name] = run(name, mixed, POINTER[name])
    td_scores[name] = evaluate(model, td_items, TRAIN_MAX_LEN, f"{name} on Typed Decisions train",
                               POINTER[name])
    torch.save(model.state_dict(), os.path.join(CKPT_DIR, f"{name}_final.pt"))
    json.dump({"curve": curves[name], "td": td_scores[name]}, open(done, "w"))
    os.remove(ckpt_path(name))
    del model
    gc.collect()
    torch.cuda.empty_cache()

## The comparison

Each arm's learning curve (C2: also each readout on dev and JevBench), then the final scores next to plain ModernBERT-base (run 8 B) and run 11's C2 on base.

In [ ]:
def chance(items):
    return round(statistics.mean(1 / len(it["names"]) for it in items), 3)


def accs(result):
    return " ".join(f"{k} {v['acc']:.3f}" for k, v in result["readouts"].items())


for name, curve in curves.items():
    print(f"{name:14s}{'point':>11s} {'train':>7s} {'dev':>7s} {'JevBench':>9s} "
          f"{'original':>9s} {'easy':>7s} {'hard':>7s}")
    for where, tr, dv, jv in curve:
        splits = " ".join(f"{jv['source'][s]['acc']:>7.3f}" for s in ("original", "easy", "hard"))
        print(f"{'':14s}{where:>11s} {tr['all']['acc']:>7.3f} {dv['all']['acc']:>7.3f} "
              f"{jv['all']['acc']:>9.3f}  {splits}")
        if POINTER[name]:
            print(f"{'':26s}dev: {accs(dv)} | JevBench: {accs(jv)}")
print(f"\n{'final':14s} {'JevBench':>9s} {'easy':>6s} {'dev':>6s} {'TD train':>9s}")
print(f"{'plain, base':14s} {0.394:>9.3f} {0.562:>6.3f} {0.457:>6.3f} {0.350:>9.3f}   (run 8 B)")
print(f"{'C2, base':14s} {0.493:>9.3f} {0.833:>6.3f} {0.471:>6.3f} {0.332:>9.3f}   (run 11, 2 passes;"
      f" pointer alone TD 0.383)")
for name, curve in curves.items():
    jv, dv, td = curve[-1][3], curve[-1][2], td_scores[name]
    print(f"{name:14s} {jv['all']['acc']:>9.3f} {jv['source']['easy']['acc']:>6.3f} "
          f"{dv['all']['acc']:>6.3f} {td['all']['acc']:>9.3f}")
    if POINTER[name]:
        print(f"{'':14s} JevBench: {accs(jv)} | dev: {accs(dv)} | TD: {accs(td)}")
print(f"chance: JevBench {chance(jev_items)} | dev {chance(dev_items)} | TD {chance(td_items)}")